In [1]:
import pandas as pd
import numpy as np

matches = pd.read_csv('matches.csv')

if 'id' in matches.columns:
    matches = matches.rename(columns={'id': 'match_id'})

cols_to_drop = ['match_type', 'player_of_match', 'target_runs', 'target_overs',
                'super_over', 'umpire1', 'umpire2', 'season', 'city', 'date',
                'toss_winner', 'toss_decision', 'result_margin', 'result', 'method',
                'team1', 'team2']
matches = matches.drop(columns=cols_to_drop, errors='ignore')  # Avoid error if columns missing

team_mapping = {
    "Royal Challengers Bengaluru": "Royal Challengers Bangalore",
    "Rising Pune Supergiant": "Rising Pune Supergiants",
    "Delhi Daredevils": "Delhi Capitals",
    "Kings XI Punjab": "Punjab Kings"
}
matches['winner'] = matches['winner'].replace(team_mapping)

venue_mapping = {
    "M Chinnaswamy Stadium": "M Chinnaswamy Stadium",
    "M.Chinnaswamy Stadium": "M Chinnaswamy Stadium",
    "M Chinnaswamy Stadium, Bengaluru": "M Chinnaswamy Stadium",
    "Punjab Cricket Association Stadium, Mohali": "Punjab Cricket Association Stadium, Mohali",
    "Punjab Cricket Association IS Bindra Stadium, Mohali": "Punjab Cricket Association Stadium, Mohali",
    "Punjab Cricket Association IS Bindra Stadium": "Punjab Cricket Association Stadium, Mohali",
    "Punjab Cricket Association IS Bindra Stadium, Mohali, Chandigarh": "Punjab Cricket Association Stadium, Mohali",
    "Wankhede Stadium": "Wankhede Stadium",
    "Wankhede Stadium, Mumbai": "Wankhede Stadium",
    "Eden Gardens": "Eden Gardens",
    "Eden Gardens, Kolkata": "Eden Gardens",
    "Sawai Mansingh Stadium": "Sawai Mansingh Stadium",
    "Sawai Mansingh Stadium, Jaipur": "Sawai Mansingh Stadium",
    "Rajiv Gandhi International Stadium, Uppal": "Rajiv Gandhi International Stadium, Hyderabad",
    "Rajiv Gandhi International Stadium, Uppal, Hyderabad": "Rajiv Gandhi International Stadium, Hyderabad",
    "Rajiv Gandhi International Stadium": "Rajiv Gandhi International Stadium, Hyderabad",
    "MA Chidambaram Stadium, Chepauk": "MA Chidambaram Stadium, Chepauk",
    "MA Chidambaram Stadium, Chepauk, Chennai": "MA Chidambaram Stadium, Chepauk",
    "MA Chidambaram Stadium": "MA Chidambaram Stadium, Chepauk",
    "Dr DY Patil Sports Academy": "Dr DY Patil Sports Academy",
    "Dr DY Patil Sports Academy, Mumbai": "Dr DY Patil Sports Academy",
    "Brabourne Stadium": "Brabourne Stadium",
    "Brabourne Stadium, Mumbai": "Brabourne Stadium",
    "Himachal Pradesh Cricket Association Stadium": "Himachal Pradesh Cricket Association Stadium",
    "Himachal Pradesh Cricket Association Stadium, Dharamsala": "Himachal Pradesh Cricket Association Stadium",
    "Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket Stadium": "Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket Stadium",
    "Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket Stadium, Visakhapatnam": "Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket Stadium",
    "Subrata Roy Sahara Stadium": "Subrata Roy Sahara Stadium",
    "Maharashtra Cricket Association Stadium": "Maharashtra Cricket Association Stadium",
    "Maharashtra Cricket Association Stadium, Pune": "Maharashtra Cricket Association Stadium",
    "Arun Jaitley Stadium": "Arun Jaitley Stadium, Delhi",
    "Arun Jaitley Stadium, Delhi": "Arun Jaitley Stadium, Delhi",
    "Feroz Shah Kotla": "Arun Jaitley Stadium, Delhi"
}

matches['venue_canonical'] = matches['venue'].map(venue_mapping).fillna(matches['venue'])

matches = matches[['match_id', 'winner', 'venue_canonical']]

deliveries = pd.read_csv('deliveries.csv')

if 'id' in deliveries.columns:
    deliveries = deliveries.rename(columns={'id': 'match_id'})

cols_deliveries = ['match_id', 'inning', 'batting_team', 'bowling_team',
                   'over', 'ball', 'total_runs', 'is_wicket']
deliveries_subset = deliveries[cols_deliveries].copy()

for col in ['batting_team', 'bowling_team']:
    deliveries_subset[col] = deliveries_subset[col].replace(team_mapping)

deliveries_subset['cum_runs'] = deliveries_subset.groupby(['match_id', 'inning'])['total_runs'].cumsum()
deliveries_subset['cum_wickets'] = deliveries_subset.groupby(['match_id', 'inning'])['is_wicket'].cumsum()
deliveries_subset['overs_completed'] = deliveries_subset['over'] + (deliveries_subset['ball'] - 1) / 6
deliveries_subset['current_run_rate'] = np.where(
    deliveries_subset['overs_completed'] == 0,
    0,
    deliveries_subset['cum_runs'] / deliveries_subset['overs_completed']
)


first_innings = deliveries_subset[deliveries_subset['inning'] == 1]
first_innings_final = first_innings.groupby('match_id')['cum_runs'].max().reset_index()
first_innings_final = first_innings_final.rename(columns={'cum_runs': 'first_innings_score'})
first_innings_final['target'] = first_innings_final['first_innings_score'] + 1

final_data = pd.merge(deliveries_subset, matches, on='match_id', how='left')

final_data = pd.merge(final_data, first_innings_final[['match_id', 'target']], on='match_id', how='left')

remaining_overs = 20 - final_data['overs_completed']
final_data['required_run_rate'] = np.where(
    (final_data['inning'] == 2) & (remaining_overs > 0),
    (final_data['target'] - final_data['cum_runs']) / remaining_overs,
    0
)
final_data['required_run_rate'] = final_data['required_run_rate'].replace([np.inf, -np.inf], 0)

final_data['win'] = (final_data['batting_team'] == final_data['winner']).astype(int)

final_data = final_data[final_data['inning'] == 2].copy()

keep_cols = ['match_id', 'inning', 'cum_runs', 'cum_wickets', 'current_run_rate',
             'required_run_rate', 'target', 'batting_team', 'bowling_team', 'venue_canonical', 'win']
final_data = final_data[keep_cols]

final_data.info()
final_data.head()


<class 'pandas.core.frame.DataFrame'>
Index: 102702 entries, 124 to 213134
Data columns (total 11 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   match_id           102702 non-null  int64  
 1   inning             102702 non-null  int64  
 2   cum_runs           102702 non-null  float64
 3   cum_wickets        102702 non-null  float64
 4   current_run_rate   102702 non-null  float64
 5   required_run_rate  102702 non-null  float64
 6   target             102702 non-null  float64
 7   batting_team       102702 non-null  object 
 8   bowling_team       102702 non-null  object 
 9   venue_canonical    102702 non-null  object 
 10  win                102702 non-null  int64  
dtypes: float64(5), int64(3), object(3)
memory usage: 9.4+ MB


,match_id,inning,cum_runs,cum_wickets,current_run_rate,required_run_rate,target,batting_team,bowling_team,venue_canonical,win
124,335982,2,1.0,0.0,0.0,11.100000,223.0,Royal Challengers Bangalore,Kolkata Knight Riders,M Chinnaswamy Stadium,0
125,335982,2,2.0,0.0,12.0,11.142857,223.0,Royal Challengers Bangalore,Kolkata Knight Riders,M Chinnaswamy Stadium,0
126,335982,2,2.0,0.0,6.0,11.237288,223.0,Royal Challengers Bangalore,Kolkata Knight Riders,M Chinnaswamy Stadium,0
127,335982,2,3.0,0.0,6.0,11.282051,223.0,Royal Challengers Bangalore,Kolkata Knight Riders,M Chinnaswamy Stadium,0
128,335982,2,4.0,0.0,6.0,11.327586,223.0,Royal Challengers Bangalore,Kolkata Knight Riders,M Chinnaswamy Stadium,0


In [2]:
from sklearn.preprocessing import LabelEncoder

In [3]:
le = LabelEncoder()

In [4]:
final_data['batting_team_encoded'] = le.fit_transform(final_data['batting_team'])
final_data['bowling_team_encoded'] = le.fit_transform(final_data['bowling_team'])
final_data['venue_encoded'] = le.fit_transform(final_data['venue_canonical'])

In [5]:
final_columns = ['inning', 'cum_runs', 'cum_wickets', 'current_run_rate',
                 'required_run_rate', 'target',
                 'batting_team_encoded', 'bowling_team_encoded', 'venue_encoded', 'win']

final_data = final_data[final_columns]

final_data.head()

,inning,cum_runs,cum_wickets,current_run_rate,required_run_rate,target,batting_team_encoded,bowling_team_encoded,venue_encoded,win
124,2,1.0,0.0,0.0,11.100000,223.0,13,6,14,0
125,2,2.0,0.0,12.0,11.142857,223.0,13,6,14,0
126,2,2.0,0.0,6.0,11.237288,223.0,13,6,14,0
127,2,3.0,0.0,6.0,11.282051,223.0,13,6,14,0
128,2,4.0,0.0,6.0,11.327586,223.0,13,6,14,0


In [6]:
from sklearn.model_selection import train_test_split

X = final_data.drop('win', axis=1)
y = final_data['win']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("Training set shape:", X_train.shape)
print("Testing set shape:", X_test.shape)

Training set shape: (82161, 9)
Testing set shape: (20541, 9)


In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "XGBoost": XGBClassifier(eval_metric='logloss', random_state=42)
}

for name, model in models.items():
    print(f"\nTraining {name}...")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    print(f"\n{name} Performance:")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
    print("Classification Report:\n", classification_report(y_test, y_pred))



Training Logistic Regression...

Logistic Regression Performance:
Accuracy: 0.777274718854973
Confusion Matrix:
 [[6915 2602]
 [1973 9051]]
Classification Report:
               precision    recall  f1-score   support

           0       0.78      0.73      0.75      9517
           1       0.78      0.82      0.80     11024

    accuracy                           0.78     20541
   macro avg       0.78      0.77      0.77     20541
weighted avg       0.78      0.78      0.78     20541


Training Random Forest...

Random Forest Performance:
Accuracy: 0.9973224283141034
Confusion Matrix:
 [[ 9491    26]
 [   29 10995]]
Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00      9517
           1       1.00      1.00      1.00     11024

    accuracy                           1.00     20541
   macro avg       1.00      1.00      1.00     20541
weighted avg       1.00      1.00      1.00     20541


Training XGBoost...

X

In [8]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import numpy as np

lr = LogisticRegression(max_iter=1000, random_state=42)
param_dist_lr = {
    'C': [0.01, 0.1, 1, 10, 100]
}

rand_search_lr = RandomizedSearchCV(
    lr,
    param_distributions=param_dist_lr,
    n_iter=5,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    random_state=42
)
rand_search_lr.fit(X_train, y_train)
print("Best parameters for Logistic Regression:", rand_search_lr.best_params_)
print("Best CV score for Logistic Regression: {:.4f}".format(rand_search_lr.best_score_))

Best parameters for Logistic Regression: {'C': 100}
Best CV score for Logistic Regression: 0.7792


In [9]:
rf = RandomForestClassifier(random_state=42)
param_dist_rf = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 5, 10],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

rand_search_rf = RandomizedSearchCV(
    rf,
    param_distributions=param_dist_rf,
    n_iter=10,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    random_state=42
)
rand_search_rf.fit(X_train, y_train)
print("\nBest parameters for Random Forest:", rand_search_rf.best_params_)
print("Best CV score for Random Forest: {:.4f}".format(rand_search_rf.best_score_))


Best parameters for Random Forest: {'n_estimators': 50, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_depth': None}
Best CV score for Random Forest: 0.9953


In [10]:
xgb = XGBClassifier(objective='binary:logistic', eval_metric='logloss',
                    use_label_encoder=False, random_state=42)
param_dist_xgb = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'min_child_weight': [1, 3, 5],
    'gamma': [0, 0.1, 0.5],
    'reg_alpha': [0, 0.1, 0.5],
    'reg_lambda': [1, 1.5, 2]
}

rand_search_xgb = RandomizedSearchCV(
    xgb,
    param_distributions=param_dist_xgb,
    n_iter=10,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    random_state=42
)
rand_search_xgb.fit(X_train, y_train)
print("\nBest parameters for XGBoost:", rand_search_xgb.best_params_)
print("Best CV score for XGBoost: {:.4f}".format(rand_search_xgb.best_score_))

/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [11:43:36] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)



Best parameters for XGBoost: {'reg_lambda': 1, 'reg_alpha': 0, 'n_estimators': 100, 'min_child_weight': 5, 'max_depth': 7, 'learning_rate': 0.2, 'gamma': 0.5}
Best CV score for XGBoost: 0.9966


In [11]:
import joblib
final_rf_model = rand_search_rf.best_estimator_
final_rf_model.fit(X_train, y_train)

y_test_pred = final_rf_model.predict(X_test)
print("Final Model Testing Accuracy: {:.2f}%".format(accuracy_score(y_test, y_test_pred)*100))
print("Final Model Classification Report (Test):")
print(classification_report(y_test, y_test_pred))
print("Final Model Confusion Matrix (Test):")
print(confusion_matrix(y_test, y_test_pred))

joblib.dump(final_rf_model, 'final_rf_model.pkl')

Final Model Testing Accuracy: 99.68%
Final Model Classification Report (Test):
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      9517
           1       1.00      1.00      1.00     11024

    accuracy                           1.00     20541
   macro avg       1.00      1.00      1.00     20541
weighted avg       1.00      1.00      1.00     20541

Final Model Confusion Matrix (Test):
[[ 9483    34]
 [   31 10993]]


['final_rf_model.pkl']